# Feature Engineering — Record Labels

Builds sparse feature matrices encoding album record label relationships.

**What it does:**
- Builds `album_labels_matrix` — label identity, weighted equally per label per album
- Builds `album_types_matrix` — MusicBrainz label type (imprint, original production, etc.), L1-normalised
- Combines both into `album_record_label_matrix` (hstack) — the single label feature block used by v3

**Why two matrices combined:** label identity and label type encode the same underlying fact (which label released the album) from two angles. Keeping them as one block in the model means the feature weight knob controls both together, which is the intended behaviour.

**Note:** `album_types_matrix` encodes *label* types, not release types (studio/live/compilation). Do not use it to detect release format — use the dedicated flag parquets for that.

**Inputs:** `data/features/album_ids.pkl`, `data/mb_album_label.parquet`

**Outputs to `data/features/`:** `album_labels_matrix.npz`, `album_types_matrix.npz`, `album_record_label_matrix.npz`

**Run after:** `01-album-artist-index.ipynb`

## Imports

In [ ]:
import pickle
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack, save_npz
from sklearn.preprocessing import normalize

DATA_DIR     = '../data'
FEATURES_DIR = f'{DATA_DIR}/features'

MIN_LABEL_ALBUMS = 10  # labels appearing on fewer albums than this are dropped

## Load Master ID Index

In [ ]:
with open(f'{FEATURES_DIR}/album_ids.pkl', 'rb') as f:
    album_ids = pickle.load(f)
album_index = pd.Index(album_ids)
n_albums = len(album_index)

print(f'Album universe: {n_albums:,}')

## Load & Prepare Label Data

In [ ]:
album_label = pd.read_parquet(f'{DATA_DIR}/mb_album_label.parquet')
album_label['label_type'] = album_label['label_type'].fillna(0.0).astype('int32')
print(f'Label rows loaded: {len(album_label):,}')

## Build Album Labels & Types Sparse Matrices

**Label matrix:** Rare labels (fewer than `MIN_LABEL_ALBUMS` albums) are dropped. The parquet has one row per `(album, label, tag)` triple, so we deduplicate to one row per `(album, label)` pair before weighting — this avoids unfairly penalising boutique labels with fewer tags. Each album's labels are then weighted equally (`1 / labels-per-album`).

**Row alignment:** `album_id` is cast to `pd.Categorical` with `categories=album_index`, which enforces the exact row ordering from the master index. Album IDs not present in the universe become NaN and are dropped.

**Types matrix:** `label_type` is a MusicBrainz integer for the kind of label (imprint, original production, etc.). Built as a binary indicator then L1-normalised row-wise, so albums with multiple label types have values that sum to 1.0.

In [ ]:
print('1. Filtering rare labels...')
label_counts  = album_label.groupby('label_id').size()
popular_labels = label_counts[label_counts >= MIN_LABEL_ALBUMS].index
label_filtered = album_label[album_label['label_id'].isin(popular_labels)].copy()

print('2. Deduplicating to one row per (album, label)...')
label_filtered = label_filtered.drop_duplicates(subset=['album_id', 'label_id'])

print('3. Normalising label weights...')
label_totals = label_filtered.groupby('album_id')['label_id'].transform('count')
label_filtered['label_weight'] = (1.0 / label_totals).astype('float32')

print('4. Generating category codes...')
label_filtered['album_id'] = pd.Categorical(label_filtered['album_id'], categories=album_index)
label_filtered = label_filtered.dropna(subset=['album_id']).copy()
label_filtered['album_code'] = label_filtered['album_id'].cat.codes
label_filtered['label_code'] = label_filtered['label_id'].astype('category').cat.codes
label_filtered['type_code']  = label_filtered['label_type'].astype('category').cat.codes

unique_label_ids = label_filtered['label_id'].astype('category').cat.categories
unique_type_ids  = label_filtered['label_type'].astype('category').cat.categories

print('5. Building sparse matrices...')
X_album_labels = csr_matrix(
    (label_filtered['label_weight'].values,
     (label_filtered['album_code'].values, label_filtered['label_code'].values)),
    shape=(n_albums, len(unique_label_ids))
)

ones = np.ones(len(label_filtered), dtype='float32')
X_album_types = csr_matrix(
    (ones,
     (label_filtered['album_code'].values, label_filtered['type_code'].values)),
    shape=(n_albums, len(unique_type_ids))
)
X_album_types = normalize(X_album_types, norm='l1', axis=1)

print(f'Labels matrix: {X_album_labels.shape}  nnz={X_album_labels.nnz:,}')
print(f'Types matrix : {X_album_types.shape}  nnz={X_album_types.nnz:,}')

## Combine into Record Label Matrix

Label identity and label type are hstacked into a single block. Both encode the same underlying fact (which label released the album) from two angles, so a single feature weight in the model controls both together.

In [ ]:
X_record_label = hstack([X_album_labels, X_album_types]).tocsr()
print(f'Record label matrix: {X_record_label.shape}  nnz={X_record_label.nnz:,}')

## Save Matrices

In [ ]:
save_npz(f'{FEATURES_DIR}/album_labels_matrix.npz',      X_album_labels)
save_npz(f'{FEATURES_DIR}/album_types_matrix.npz',       X_album_types)
save_npz(f'{FEATURES_DIR}/album_record_label_matrix.npz', X_record_label)

print(f'album_labels_matrix      : {X_album_labels.shape}')
print(f'album_types_matrix       : {X_album_types.shape}')
print(f'album_record_label_matrix: {X_record_label.shape}')
print('Done.')